# Reviewed rewriter data check for Colab

This notebook uses `reviewed-v2/` in the private Hugging Face training repository. Add `HF_TOKEN` to Colab Secrets with read access. It checks all three splits and prepares 16 training examples. It loads no model weights and runs no training. For the later training proof, select **Runtime > Change runtime type > L4 GPU**. This data check also runs on a CPU.


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==5.21.0",
        "huggingface_hub==1.33.0",
        "jinja2==3.1.6",
    ]
)

In [ ]:
import hashlib
import json
from collections import Counter
from pathlib import Path

from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

repo = "jayshah5696/finetuning-labs-rewriter-training"
prefix = "reviewed-v2"
token = userdata.get("HF_TOKEN")
revision = HfApi(token=token).repo_info(repo, repo_type="dataset").sha


def download(name):
    path = hf_hub_download(
        repo, f"{prefix}/{name}", repo_type="dataset", revision=revision, token=token
    )
    return Path(path).read_bytes()


manifest = json.loads(download("manifest.json"))
payloads = {}
for name, expected in manifest["files"].items():
    data = download(name)
    assert hashlib.sha256(data).hexdigest() == expected, name
    payloads[name] = data
quality = json.loads(payloads["quality.json"])
support = json.loads(payloads["supporting.json"])
assert quality["all_exact_rows_opus_pass"] is True
assert quality["human_reviewed"] is False
print(
    {
        "repo": repo,
        "immutable_revision": revision,
        "prefix": prefix,
        "splits": quality["splits"],
        "human_labels": False,
    }
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "LiquidAI/LFM2.5-230M", revision="40cb2ad3b3044d5a41eee083a6103c8b523afa45"
)
dimensions = {
    "candidate_input",
    "output_quality",
    "pair_facts",
    "pair_intent",
    "pair_literals",
    "pair_mode",
}
split_names = ("train", "dev", "validation")
splits, prepared, rows_by_split, meta_by_split = {}, {}, {}, {}
review_reports = {
    name: json.loads(payloads[name]) for name in {item["opus_review"]["report"] for item in support}
}
for split in split_names:
    rows = [json.loads(line) for line in payloads[f"data/{split}.jsonl"].splitlines()]
    guides = [json.loads(line) for line in payloads[f"sft/{split}.jsonl"].splitlines()]
    meta = [item for item in support if item["split"] == split]
    assert len(rows) == len(guides) == len(meta) == quality["splits"][split]
    prepared[split], rows_by_split[split], meta_by_split[split] = [], guides, meta
    for row, guide, item in zip(rows, guides, meta, strict=True):
        assert set(row) == {"input", "output", "mode", "checks"}
        assert set(guide) == {"system", "user", "assistant"}
        assert guide["assistant"] == row["output"]
        encoded = json.dumps(
            row, ensure_ascii=False, sort_keys=True, separators=(",", ":")
        ).encode()
        assert hashlib.sha256(encoded).hexdigest() == item["row_sha256"]
        assert not item["mechanical_issues"] and item["human_reviewed"] is False
        assert set(item["decisions"]) == {"labeling_engine." + k for k in dimensions}
        review = item["opus_review"]
        report = review_reports[review["report"]]
        matched = [
            r
            for r in report["reviews"]
            if (r["id"], r["row_sha256"]) == (item["id"], item["row_sha256"])
        ]
        assert len(matched) == 1
        assert all(
            matched[0][key] == "pass"
            for key in ("input_verdict", "output_verdict", "pair_verdict", "verdict")
        )
        header = [f"MODE: {row['mode']}", f"LANGUAGE: {item['language']}"]
        if item["context"].get("profile"):
            header.append("PROFILE: " + item["context"]["profile"])
        if item["context"].get("dictionary"):
            header.append("DICTIONARY: " + ", ".join(item["context"]["dictionary"]))
        assert guide["user"] == "\n".join(header + ["INPUT:", row["input"]])
        if item.get("identity"):
            assert row["input"] == row["output"]
        if item["length"] == "long":
            assert 1000 <= len(tokenizer.encode(row["input"], add_special_tokens=False)) <= 2000
        messages = [
            {"role": "system", "content": guide["system"]},
            {"role": "user", "content": guide["user"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=False
        )
        full = tokenizer.apply_chat_template(
            messages + [{"role": "assistant", "content": guide["assistant"]}],
            tokenize=True,
            return_dict=False,
        )
        assert full[: len(prompt)] == prompt
        assert len(full) <= 4096 and len(prompt) + 2048 <= 4096
        end = len(prompt) + full[len(prompt) :].index(tokenizer.eos_token_id) + 1
        assert len(prompt) < end - 1 and end - len(prompt) <= 2048
        assert not tokenizer.decode(full[end:]).strip()
        labels = [-100] * len(prompt) + full[len(prompt) : end] + [-100] * (len(full) - end)
        assert sum(label != -100 for label in labels) == item["tokens"]["assistant_tokens"]
        prepared[split].append({"input_ids": full, "labels": labels})
    splits[split] = dict(Counter(row["mode"] for row in rows))

groups = {split: {m["source_group"] for m in meta_by_split[split]} for split in split_names}
for a, b in [("train", "dev"), ("train", "validation"), ("dev", "validation")]:
    assert groups[a].isdisjoint(groups[b]), (a, b)
assert sum(len(v) for v in prepared.values()) == 166
print(
    {
        "checks": "passed",
        "rows_by_mode": splits,
        "long_pairs": quality["long_pairs"],
        "max_chat_tokens": max(len(r["input_ids"]) for v in prepared.values() for r in v),
    }
)

Use `train` for fitting and `dev` for checkpoint selection. Keep `validation` for the stock-versus-tuned comparison. It is development validation, not an unseen final test. The original 68-case files and historical scores are preserved. Rerun the stock model on this v2 validation bundle before comparing a fine-tune.

The next cell selects four training examples per mode. It includes an unchanged short input, an edited short input, and two long inputs. Where available, one long input is unchanged. These examples are for a memorization check, so success on them does not measure generalization.


In [ ]:
train_meta = meta_by_split["train"]
selected = []
for mode in ("/slack", "/transcribe", "/prompt", "/polish"):
    candidates = [
        i
        for i, guide in enumerate(rows_by_split["train"])
        if guide["user"].startswith(f"MODE: {mode}\n")
    ]
    for identity in (True, False):
        selected.append(
            next(
                i
                for i in candidates
                if train_meta[i]["length"] == "short"
                and bool(train_meta[i].get("identity")) is identity
            )
        )
    long_indices = [i for i in candidates if train_meta[i]["length"] == "long"]
    long_indices.sort(key=lambda i: not bool(train_meta[i].get("identity")))
    assert len(long_indices) >= 2
    selected.extend(long_indices[:2])
assert len(set(selected)) == 16
proof_path = Path("rewriter-memorization-16.jsonl")
proof_path.write_text(
    "".join(json.dumps(rows_by_split["train"][i], ensure_ascii=False) + "\n" for i in selected)
)
report = {
    "repo": repo,
    "revision": revision,
    "prefix": prefix,
    "splits": quality["splits"],
    "file_hashes": manifest["files"],
    "assistant_only_loss_checked": True,
    "all_exact_rows_opus_pass": True,
    "independent_human_labels": False,
    "selected_training_ids": [train_meta[i]["id"] for i in selected],
    "student_training_run": False,
}
Path("rewriter-data-readiness.json").write_text(json.dumps(report, indent=2) + "\n")
print(
    {
        "proof_rows": len(selected),
        "proof_file": str(proof_path),
        "readiness_receipt": "rewriter-data-readiness.json",
        "training_run": False,
    }
)